# 🚀 FaceKey Collector — Kaggle
### Dedicated Worker: `kaggle-worker-1` (3 Parallel Workers, 1 Face Only, Turbo 150+ FPS)
- **Queue**: `youtube_urls_kaggle.txt` (117 Dedicated Videos — Zero Overlap)
- **Storage**: Packages 5-session `.tar.gz` chunks into `/kaggle/working/FaceKeyDataset/chunks/`.
- **Local PC Data Used**: **0 MB**.

In [ ]:
# ==============================================================================
# ⚡ 1-CLICK INSTANT RUNNER: KAGGLE WORKER
# ==============================================================================
import os, sys, shutil

# 1. Working directories
DRIVE_DIR = "/kaggle/working/FaceKeyDataset"
for folder in ("chunks", "checkpoints", "locks", "data"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)

# 2. Update to latest GitHub repository code (Self-Healing)
%cd /kaggle/working
APP_DIR = "/kaggle/working/FaceKeyAnimation"
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    print("[*] Updating FaceKeyAnimation to latest GitHub main...")
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print("[*] Fresh clone of FaceKeyAnimation from GitHub...")
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Dedicated Queue for Kaggle (Zero Overlap with Colab)
urls_file = os.path.join(APP_DIR, "sessions", "youtube_urls_kaggle.txt")

# 5. Launch Collector (3 Workers across 4 CPU cores, 1 Face Only, Turbo Mode)
print("\n[*] Launching FaceKey Turbo Multi-Worker Kaggle Collector (kaggle-worker-1)...")
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "kaggle-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p \
    --num-workers 3


---
# 📦 KAGGLE STORAGE MANAGEMENT & CLOUD UPLOAD (0 MB LOCAL DATA)
> [!TIP]
> Check your disk usage and pending chunks below anytime. When you run an upload, local files are automatically purged to keep Kaggle's disk clean.

In [ ]:
# 📊 Check Kaggle Disk Space & Pending Chunks
%cd /kaggle/working/FaceKeyAnimation
!python scripts/cloud_drive_uploader.py --method status


In [ ]:
# ☁️ Direct Cloud-to-Cloud Upload to your 5 TB Google Drive (via Service Account)
SERVICE_ACCOUNT_JSON = "/kaggle/working/service_account.json"
GOOGLE_DRIVE_FOLDER_ID = "YOUR_GOOGLE_DRIVE_FOLDER_ID"

!python scripts/cloud_drive_uploader.py \
    --method gdrive \
    --service-account "{SERVICE_ACCOUNT_JSON}" \
    --folder-id "{GOOGLE_DRIVE_FOLDER_ID}"


In [ ]:
# 🚀 Zero-GCP HuggingFace Bridge (Free Unlimited Private Dataset)
HF_TOKEN = "YOUR_HF_WRITE_TOKEN"
HF_REPO = "YOUR_USERNAME/facekey-chunks"

!python scripts/cloud_drive_uploader.py \
    --method hf \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}"
